In [1]:
import os

BASE_DIR = "/kaggle/working/spectral_v2"
DIRS = {
    "generations": f"{BASE_DIR}/generations", "extractions": f"{BASE_DIR}/extractions",
    "features": f"{BASE_DIR}/features", "results": f"{BASE_DIR}/results", "logs": f"{BASE_DIR}/logs",
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

In [2]:
import os
print(os.listdir("/kaggle/input/notebooks/aishidev"))

['final-spectral-research-project-arc-features', 'final-spectral-research-project-gsm8k-features']


In [3]:
import shutil

os.makedirs(f"{DIRS['features']}", exist_ok=True)

GSM8K_SOURCE = "/kaggle/input/notebooks/aishidev/final-spectral-research-project-gsm8k-features/spectral_v2"
ARC_SOURCE = "/kaggle/input/notebooks/aishidev/final-spectral-research-project-arc-features/spectral_v2/features"

shutil.copytree(GSM8K_SOURCE, BASE_DIR, dirs_exist_ok=True)
shutil.copytree(ARC_SOURCE, DIRS["features"], dirs_exist_ok=True)

print("Files in features dir:", os.listdir(DIRS["features"]))
print("Files in results dir:", os.listdir(DIRS["results"]))

Files in features dir: ['metrics_per_example_layer_gsm8k_NORMALIZED.csv', 'metrics_per_example_layer_arc_NORMALIZED.csv', 'arc_step9_master_table.csv', 'metrics_per_example_layer_gsm8k.csv', 'step9_master_table.csv', 'metrics_per_example_layer_arc.csv']
Files in results dir: ['gsm8k_zoomed_boxplots_spectral_entropy.png', 'gsm8k_zoomed_boxplots_smoothness.png', 'gsm8k_length_confound_check.csv', 'gsm8k_raw_effect_sizes.csv', 'gsm8k_zoomed_boxplots_hfer.png', 'gsm8k_sink_rank_confounds_fixed.csv', 'step9_classifier_comparison.csv', 'gsm8k_NORMALIZED_length_confound_check.csv', 'gsm8k_label_review.csv', 'gsm8k_trajectory_plots.png', 'gsm8k_zoomed_boxplots_fiedler.png', 'gsm8k_layer16_sink_rank_confounds.csv']


In [4]:
# ============================================================
# CLEANUP FIX 1 — GSM8K layer 16 (spectral_entropy, normalized
# Laplacian survivor): leakage-safe classifier test to see if
# it adds predictive power beyond confounds.
# No GPU needed.
# ============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegressionCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# --- Load and merge everything needed for layer 16 ---
features_norm_df = pd.read_csv(f"{DIRS['features']}/metrics_per_example_layer_gsm8k_NORMALIZED.csv")
sink_rank_df = pd.read_csv(f"{DIRS['results']}/gsm8k_layer16_sink_rank_confounds.csv")

layer16 = features_norm_df[features_norm_df["layer"] == 16][["example_id", "spectral_entropy", "response_len", "is_correct"]]
master = layer16.merge(sink_rank_df[["example_id", "sink_mass", "effective_rank"]], on="example_id")
master = master.rename(columns={"sink_mass": "sink_mass_L16", "effective_rank": "effective_rank_L16"})

print(f"Master table: {master.shape}")
print(f"Correct: {(master['is_correct']==True).sum()}, Incorrect: {(master['is_correct']==False).sum()}")

CONFOUND_COLS = ["response_len", "sink_mass_L16", "effective_rank_L16"]
SPECTRAL_COL = ["spectral_entropy"]

y = master["is_correct"].astype(int).values

N_OUTER_FOLDS = 5
skf = StratifiedKFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=42)

def residualize_fold(train_df, test_df, spectral_cols, confound_cols):
    train_resid = train_df[spectral_cols].copy()
    test_resid = test_df[spectral_cols].copy()
    X_train_confound = train_df[confound_cols].values
    X_test_confound = test_df[confound_cols].values
    for col in spectral_cols:
        reg = LinearRegression()
        reg.fit(X_train_confound, train_df[col].values)
        train_resid[col] = train_df[col].values - reg.predict(X_train_confound)
        test_resid[col] = test_df[col].values - reg.predict(X_test_confound)
    return train_resid, test_resid

FEATURE_SETS = {
    "confound_only": CONFOUND_COLS,
    "layer16_spectral_entropy_only": SPECTRAL_COL,
    "confound_plus_layer16": CONFOUND_COLS + SPECTRAL_COL,
}

results = {name: [] for name in FEATURE_SETS}

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(master, y)):
    train_df = master.iloc[train_idx].reset_index(drop=True)
    test_df = master.iloc[test_idx].reset_index(drop=True)
    y_train, y_test = y[train_idx], y[test_idx]

    for name, cols in FEATURE_SETS.items():
        if name == "confound_only":
            X_train = train_df[cols].values
            X_test = test_df[cols].values
        elif name == "layer16_spectral_entropy_only":
            # residualize spectral_entropy against confounds, fit on train only
            train_resid, test_resid = residualize_fold(train_df, test_df, SPECTRAL_COL, CONFOUND_COLS)
            X_train = train_resid.values
            X_test = test_resid.values
        else:  # confound_plus_layer16
            train_resid, test_resid = residualize_fold(train_df, test_df, SPECTRAL_COL, CONFOUND_COLS)
            X_train = np.hstack([train_df[CONFOUND_COLS].values, train_resid.values])
            X_test = np.hstack([test_df[CONFOUND_COLS].values, test_resid.values])

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        clf = LogisticRegressionCV(Cs=10, cv=3, penalty="l2", max_iter=2000, scoring="roc_auc", random_state=42)
        clf.fit(X_train_scaled, y_train)

        y_pred_proba = clf.predict_proba(X_test_scaled)[:, 1]
        fold_auc = roc_auc_score(y_test, y_pred_proba)
        results[name].append(fold_auc)

    print(f"Fold {fold_idx + 1}/{N_OUTER_FOLDS} done.")

print("\n" + "=" * 60)
print("RESULTS: Does layer-16 spectral entropy add value beyond confounds?")
print("=" * 60)
for name in FEATURE_SETS:
    aucs = results[name]
    print(f"{name:35s}: AUC = {np.mean(aucs):.3f} +/- {np.std(aucs):.3f}")

baseline_auc = np.mean(results["confound_only"])
combined_auc = np.mean(results["confound_plus_layer16"])
print(f"\nConfound-only baseline: {baseline_auc:.3f}")
print(f"Confound + layer16 spectral entropy: {combined_auc:.3f}")
print(f"Difference: {combined_auc - baseline_auc:+.3f}")

Master table: (150, 6)
Correct: 65, Incorrect: 85
Fold 1/5 done.
Fold 2/5 done.
Fold 3/5 done.
Fold 4/5 done.
Fold 5/5 done.

RESULTS: Does layer-16 spectral entropy add value beyond confounds?
confound_only                      : AUC = 0.715 +/- 0.076
layer16_spectral_entropy_only      : AUC = 0.663 +/- 0.099
confound_plus_layer16              : AUC = 0.710 +/- 0.039

Confound-only baseline: 0.715
Confound + layer16 spectral entropy: 0.710
Difference: -0.005


In [5]:
# ============================================================
# CLEANUP FIX 2 — ARC static-final classifier result:
# permutation test to check if 0.640 AUC is actually
# distinguishable from what random label-shuffling produces.
# This reconciles the classifier result with BH correction's
# "0 of 96 significant" finding.
# No GPU needed.
# ============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

master = pd.read_csv(f"{DIRS['features']}/arc_step9_master_table.csv")
STATIC_FINAL_COLS = [c for c in master.columns if c.endswith("_final")]
print(f"Static final columns: {STATIC_FINAL_COLS}")

X = master[STATIC_FINAL_COLS].values
y_real = master["is_correct"].astype(int).values

def run_cv_auc(X, y, seed=42):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    aucs = []
    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        clf = LogisticRegressionCV(Cs=10, cv=3, penalty="l2", max_iter=2000, scoring="roc_auc", random_state=42)
        clf.fit(X_train_scaled, y_train)
        y_pred = clf.predict_proba(X_test_scaled)[:, 1]
        aucs.append(roc_auc_score(y_test, y_pred))
    return np.mean(aucs)

# --- Real result ---
real_auc = run_cv_auc(X, y_real)
print(f"\nReal AUC (static final-layer features): {real_auc:.3f}")

# --- Permutation test: shuffle labels many times, rerun ---
N_PERMUTATIONS = 200
np.random.seed(0)
permuted_aucs = []

print(f"\nRunning {N_PERMUTATIONS} permutations (shuffled labels)...")
for i in range(N_PERMUTATIONS):
    y_shuffled = np.random.permutation(y_real)
    perm_auc = run_cv_auc(X, y_shuffled, seed=42)
    permuted_aucs.append(perm_auc)
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{N_PERMUTATIONS} done")

permuted_aucs = np.array(permuted_aucs)

print(f"\n{'='*60}")
print("PERMUTATION TEST RESULTS")
print(f"{'='*60}")
print(f"Real AUC: {real_auc:.3f}")
print(f"Permuted AUCs: mean={permuted_aucs.mean():.3f}, std={permuted_aucs.std():.3f}")
print(f"Permuted AUC range: [{permuted_aucs.min():.3f}, {permuted_aucs.max():.3f}]")

# p-value: fraction of permuted AUCs >= real AUC
p_value = (permuted_aucs >= real_auc).mean()
print(f"\nPermutation p-value: {p_value:.4f}")
print(f"(fraction of {N_PERMUTATIONS} random-label runs that matched or beat the real result)")

if p_value < 0.05:
    print("\n>>> The real AUC is significantly higher than chance-level permutations. <<<")
else:
    print("\n>>> The real AUC is NOT significantly different from random label shuffling. <<<")
    print(">>> This is consistent with BH correction finding no significant underlying effects. <<<")

Static final columns: ['fiedler_final', 'spectral_entropy_final', 'hfer_final', 'smoothness_final']

Real AUC (static final-layer features): 0.611

Running 200 permutations (shuffled labels)...
  50/200 done
  100/200 done
  150/200 done
  200/200 done

PERMUTATION TEST RESULTS
Real AUC: 0.611
Permuted AUCs: mean=0.504, std=0.066
Permuted AUC range: [0.295, 0.663]

Permutation p-value: 0.0550
(fraction of 200 random-label runs that matched or beat the real result)

>>> The real AUC is NOT significantly different from random label shuffling. <<<
>>> This is consistent with BH correction finding no significant underlying effects. <<<
